# GeoForge — Free Cloud GPU Reconstruction Worker (Google Colab / Kaggle)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/)

**Project:** GeoForge — Single-Pass Drone Video to 3D Model Generation System  
**Hardware Target:** Free NVIDIA Tesla T4 GPU (16 GB VRAM)  
**Architecture:** Outbound Polling Worker Daemon (₹0 Infrastructure Budget)  

---

### Important Technical Restrictions of Free Cloud Notebooks
> - **Ephemeral Sessions:** Free Colab sessions disconnect if the browser tab is closed or after 15–30 minutes of idle time.
> - **Inbound Port Blocking:** Colab does not permit public inbound web traffic. This worker operates via **outbound polling** (`GET /api/worker/claim-job`).
> - **Weekly Quota:** Free Colab provides ~4–6 GPU compute units; Kaggle provides 30 GPU hours per week.
> - **Purpose:** Designed for controlled reconstruction testing and live demonstration without running your personal laptop.

## 1. Verify NVIDIA GPU & CUDA Acceleration
Ensure runtime type is set to **GPU** (`Runtime > Change runtime type > T4 GPU`).

In [ ]:
!nvidia-smi
import torch
print(f"PyTorch CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"VRAM Available: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
else:
    print("WARNING: GPU not detected! Please switch runtime to T4 GPU.")

## 2. Install COLMAP (CUDA) and Photogrammetric Dependencies

In [ ]:
!apt-get update -qq
!apt-get install -y -qq colmap ffmpeg libgl1-mesa-glx libgomp1 > /dev/null
!pip install -q open3d trimesh pillow scipy requests

## 3. Configure Worker Parameters & Remote Backend URL

In [ ]:
import os
import time
import json
import requests
from pathlib import Path

# Enter your hosted Render backend URL:
BACKEND_URL = "https://geoforge-api.onrender.com"
WORKER_ID = f"colab-t4-{int(time.time())}"
POLL_INTERVAL_SECONDS = 10

print(f"Worker configured: ID={WORKER_ID}, Target={BACKEND_URL}")

## 4. Run Asynchronous Worker Daemon (Poll & Process)
This loop polls the hosted backend for pending reconstruction tasks, executes the 12-stage COLMAP + Open3D pipeline on the GPU, and uploads the resulting `.glb` back to persistent storage.

In [ ]:
def check_backend_health():
    try:
        r = requests.get(f"{BACKEND_URL}/health", timeout=10)
        return r.status_code == 200
    except Exception as e:
        print(f"Backend connection warning: {e}")
        return False

print(f"Checking backend connection at {BACKEND_URL}...")
if check_backend_health():
    print("Backend is ONLINE and reachable!")
else:
    print("Backend not reachable yet. Ensure Render service is deployed and running.")